In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","billing","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
# billing_id
df_silver = df_silver.withColumn(
    "billing_id",
    F.trim(F.col("billing_id"))
#  resident_id
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
#  facility_id
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
#  billing_date
).withColumn(
    "billing_date",
    F.trim(F.col("billing_date"))
#  amount_aud
).withColumn(
    "amount_aud",
    F.trim(F.col("amount_aud"))
#  billing_type
).withColumn(
    "billing_type",
    F.trim(F.col("billing_type"))
#  payment_status
).withColumn(
    "payment_status",
    F.trim(F.col("payment_status"))
#  payment_date
).withColumn(
    "payment_date",
    F.trim(F.col("payment_date"))
#  invoice_number
).withColumn(
    "invoice_number",
    F.trim(F.col("invoice_number"))
#  notes
).withColumn(
    "notes",
    F.trim(F.col("notes"))
#  created_at
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# billing_id
filt_df = df_silver.filter(col("billing_id").rlike("^//BIL"))
display(filt_df)
display(df_silver)

In [0]:
#  resident_id
filt_df = df_silver.filter(col("resident_id").rlike("^//RES"))
display(filt_df)

In [0]:
#  facility_id
filt_df = df_silver.filter(col("facility_id").rlike("^//FAC"))
display(filt_df)

In [0]:
# billing_date
from pyspark.sql.functions import col,when
df_silver = df_silver.withColumn("billing_date",when(~col("billing_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),None).otherwise(col("billing_date")))

df_silver = df_silver.withColumn("billing_date",when(col("billing_date")==("9999-99-99"),None).otherwise(col("billing_date")))

dup = df_silver.filter(col("billing_date").rlike("9999-99-99"))
display(dup)

dup = df_silver.filter(~col("billing_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(dup)

df_filt = df_silver.filter(col("billing_date").rlike("not-a-date"))
display(df_filt)

df_silver = df_silver.withColumn("billing_date",when(~df_silver["billing_date"].rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$"),None).otherwise(col("billing_date")))

df_filt = df_silver.filter(~df_silver["billing_date"].rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$"))
display(df_filt)

from pyspark.sql.functions import col, to_timestamp
df_silver = df_silver.withColumn(
    "billing_date",
    to_timestamp(col("billing_date"), "yyyy-MM-dd HH:mm:ss")  
)
display(df_silver)


In [0]:
#  amount_aud
from pyspark.sql.functions import col,when,abs
df_silver = df_silver.withColumn("amount_aud",abs(col("amount_aud")))
dup = df_silver.filter(col("amount_aud").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(dup)
df_filt = df_silver.filter(df_silver["amount_aud"].rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$"))
# display(df_filt)
display(df_silver)

In [0]:
#  billing_type
from pyspark.sql.functions import col,when,abs,trim,upper
df_silver = df_silver.withColumn("billing_type",upper(trim(col("billing_type"))))

dup = df_silver.groupBy("billing_type").count()
# display(dup)

reply_asd = {"NAN" : "NOT PROVIDE",
"null" : "NOT PROVIDE",
"NULL" : "NOT PROVIDE",
"UNKNOWN" : "NOT PROVIDE",
"N/A" : "NOT PROVIDE",
"#N/A" : "NOT PROVIDE",
"NONE" : "NOT PROVIDE",
"" : "NOT PROVIDE"
}
df_silver = df_silver.replace(reply_asd,subset = ["billing_type"])
df_silver = df_silver.fillna({"billing_type":"NOT PROVIDE"})
dup = df_silver.groupBy("billing_type").count()
display(dup)

In [0]:
#  payment_status
from pyspark.sql.functions import col,when,abs,trim,upper

df_silver = df_silver.withColumn("payment_status",upper(trim(col("payment_status"))))

dup = df_silver.groupBy("payment_status").count()
display(dup)

In [0]:
#  payment_date
from pyspark.sql.functions import col,when,abs,trim,upper
df_silver = df_silver.withColumn("payment_date",trim(col("payment_date")))
dup = df_silver.groupBy("payment_date").count().filter(col("count")>100)
# display(dup)

replace_hd = {"UNKNOWN" : None,
"null" : None,
"#N/A" : None,
"NULL" : None,
"N/A" : None,
""   : None,
"NONE" : None,
"null" : None,
"NaN" : None}
df_silver = df_silver.replace(replace_hd,subset = ["payment_date"])

dup = df_silver.groupBy("payment_date").count().filter(col("count")>100)
display(dup)
from pyspark.sql.functions import col, to_timestamp
df_silver = df_silver.withColumn(
    "payment_date",
    to_timestamp(col("payment_date"), "yyyy-MM-dd HH:mm:ss")  
)
display(df_silver)

In [0]:
#  invoice_number
from pyspark.sql.functions import col,when,abs,trim,upper
dup = df_silver.filter(col("invoice_number").rlike("^//INV"))
display(dup)

In [0]:
#  notes
from pyspark.sql.functions import col,when,abs,trim,upper,initcap
df_silver = df_silver.withColumn("notes",initcap(trim(col("notes"))))
dup = df_silver.groupBy("notes").count().filter(col("count")>1)
# display(dup)

reply_dg = {"Nan" : "Unknown",
"None" : "Unknown",
"N/a" : "Unknown",
"#n/a" : "Unknown",
"Unknown" : "Unknown",
"Null" : "Unknown",
"" : "Unknown"
}
df_silver = df_silver.replace(reply_dg,subset = ["notes"])
df_silver = df_silver.fillna({"notes":"Unknown"})
dup = df_silver.groupBy("notes").count().filter(col("count")>1)
display(dup)

In [0]:
#  created_at
from pyspark.sql.functions import col,when,abs,trim,upper,initcap

# df_silver = df_silver.withColumn("notes",initcap(trim(col("notes"))))
dup = df_silver.groupBy("created_at").count().filter(col("count")>10)
display(dup)
df_silver = df_silver.withColumn("created_at",to_timestamp(col("created_at"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)


#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")